# Strong — Data Exploration
Downloads the workout CSV from Google Drive and runs the extractor logic.

In [ ]:
import sys
import tempfile
from pathlib import Path

import gdown
import polars as pl
from dotenv import dotenv_values

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

env = dotenv_values(ROOT / '.env')
DRIVE_URL = env['DRIVE_SHARE_URL']

In [ ]:
tmp = tempfile.mkdtemp()
gdown.download_folder(url=DRIVE_URL, output=tmp, use_cookies=False, quiet=False)

strong_dir = Path(tmp) / 'strong'
csv_files = list(strong_dir.glob('*.csv'))
print(f'Found {len(csv_files)} CSV(s):', [f.name for f in csv_files])

In [ ]:
# Peek at raw columns before extraction
import io
raw = pl.read_csv(io.BytesIO(csv_files[0].read_bytes()), separator=';', infer_schema_length=5)
print('Columns:', raw.columns)
raw.head(5)

In [ ]:
from pipeline.extractors.strong import _read_csv

frames = [_read_csv(f.read_bytes()) for f in csv_files]
df = pl.concat(frames)
print(f'{len(df)} rows, {df["date"].min()} → {df["date"].max()}')
df

In [ ]:
# Workouts per week
df.with_columns(
    pl.col('date').dt.truncate('1w').alias('week')
).group_by('week').agg(pl.len().alias('sessions')).sort('week')

In [ ]:
# Most common workouts
df.group_by('category').agg(
    pl.len().alias('sessions'),
    pl.col('value').mean().round(1).alias('avg_duration_min')
).sort('sessions', descending=True)